# Youtube Data API: Videos
https://developers.google.com/youtube/v3/docs/videos

## Contenido:

* [Librerias](*Librerias)
* [Videos](*Videos)


<a name="Librerias"></a>
## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
import math
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

#### https://www.youtube.com/playlist?list=
#### https://www.youtube.com/channel/
#### https://www.youtube.com/watch?v=

## Videos

### Ejecucion

#### Parametros constantes

In [ ]:
URIv='https://www.googleapis.com/youtube/v3/videos?key='

# ================== Parametros Requeridos ==================

part = 'snippet,contentDetails,id,recordingDetails,statistics,topicDetails,status' # Parametro obligatorio

#### Parametros variables

In [ ]:
# ================== Filtros (especifique exactamente uno de los siguientes parámetros) ==================

id = 'mVlgrlDz744,' # ID de videos (Solo se puede un maximo de 50 ids)

#### Solicitud

In [ ]:
r = requests.get(URIv+key+'&part='+part+'&id='+id)

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d.keys()

In [ ]:
d

In [ ]:
d['kind']

In [ ]:
d['etag']

In [ ]:
d['pageInfo']

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data=pd.json_normalize(items)
data.info()

In [ ]:
data

### Algoritmo en Produccion

#### Importando Videos

In [ ]:
# ================== OJO modificar para cada serie ==================

path='/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/VideosSupremo/\
Narcos/2022-09-13/VideosRelevantesxGumelab.csv' 

# ================== OJO modificar para cada serie ==================

Videos = pd.read_csv(path)
Videos

#### Formato Lista de IDs de Videos

In [ ]:
iz=0
de=50
L=list(Videos['idRecursoVD'])
ent=math.ceil(len(L)/50)
ListaIDs=[]
for i in range(0,ent):
  stringIDs=','.join(L[iz:de])
  iz=iz+50
  de=de+50
  ListaIDs.append(stringIDs)
len(ListaIDs)

In [ ]:
ListaIDs

#### Parametros constantes

In [ ]:
URIv='https://www.googleapis.com/youtube/v3/videos?key='

# ================== Parametros Requeridos ==================

part = 'snippet,contentDetails,id,recordingDetails,statistics,topicDetails,status' # Parametro obligatorio
fields = 'pageInfo(*),nextPageToken(*),prevPageToken(*),\
items(id,snippet(publishedAt,channelId,title,description,channelTitle,tags,categoryId,liveBroadcastContent,defaultLanguage),\
statistics(*))'

#### Parametros variables

In [ ]:
# ================== Filtros (especifique exactamente uno de los siguientes parámetros) ==================

id = ListaIDs # Lista de ids de videos

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Videos/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'VIDEOS-RelevantesxGumelab'+str(datetime.now())

#### Constantes

In [ ]:
solicitud = 0 # Inicio de las solicitudes
espera = 1 # Valor de retardo para las solicitudes
solicitudmax = 7000 # Numero de solicitudes maximas (el limite es 10000)

# ======================= OJO Cambiar diariamente =======================

ithid = 0 # i-esimo id

# ======================= OJO Cambiar diariamente =======================

#### Ejecucion

In [ ]:
Datac = pd.DataFrame() # Dataframe de Resultados Totales
Metadatac = pd.DataFrame(columns=['Solicitud','ListaIDs','FechaEjecucion'\
                                  ,'Respuesta','ResultadosTotal','Resultadosxpagina']) # Dataframe de Informacion de Solicitudes

for id in ListaIDs[ithid:]: # For de conjuntos de 50 id's
  
  solicitud=solicitud+1
  
  if (solicitud == solicitudmax+1):
    ptoPartida = [] # Punto de partida para volver a comenzar el motor de busqueda en produccion.
    break

  time.sleep(espera) # Tiempo de espera por cada solicitud
  r = requests.get(URIv+key+'&part='+part+'&fields='+fields+'&id='+id) # Solicitud

  print('solicitud',solicitud,r.reason)

  if (r.reason == 'OK'): # Vemos si la solicitud fue exitosa
    
    d = dict(r.json())
    items = d.get('items')
    Data=pd.json_normalize(items).assign(Solicitud=solicitud)
    Datac=pd.concat([Datac, Data], ignore_index=True)  
    Metadatac = Metadatac.append({'Solicitud':solicitud,'ListaIDs':id,'FechaEjecucion':str(datetime.now())\
                                  ,'Respuesta':r.reason,'ResultadosTotal':d['pageInfo']['totalResults']\
                                  ,'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index=True)
  
  else: # Vemos si la solicitud no fue exitosa    
    print("No fue exitosa la solicitud", solicitud)

print("Videos Terminados")

In [ ]:
Datac.info()

In [ ]:
Datac

In [ ]:
Metadatac.info()

In [ ]:
Metadatac

#### Uniendo columnas de importancia

In [ ]:
mergedinner = pd.merge(left=Datac,right=Videos, left_on='id', right_on='idRecursoVD').drop(['idRecursoVD'], axis=1)
mergedinner.info()

#### Estructurar y dar formato a Dataframe

In [ ]:
columnas={'id':'idVideo','snippet.publishedAt':'FechaPublicacionVideo','snippet.channelId':'idCanalVideo','snippet.title':'TituloVideo'\
          ,'snippet.description':'DescripcionVideo','snippet.channelTitle':'TituloCanalVideo','snippet.categoryId':'CategoriaVideo'\
          ,'snippet.liveBroadcastContent':'TransmisionVivoVideo','statistics.viewCount':'VistasVideo','statistics.likeCount':'LikesVideo'\
          ,'statistics.favoriteCount':'FavoritoVideo','statistics.commentCount':'ComentariosVideo','snippet.tags':'EtiquetasVideo','Solicitud':'SolicitudVideo'\
          ,'Consulta':'ConsultaVideo','Pais':'PaisVideo'}

columnasorden=['idVideo','TituloVideo','DescripcionVideo','FechaPublicacionVideo','CategoriaVideo','VistasVideo','LikesVideo','FavoritoVideo','ComentariosVideo'\
               ,'EtiquetasVideo','TransmisionVivoVideo','idCanalVideo','TituloCanalVideo','ConsultaVideo','PaisVideo','SolicitudVideo']

mergedinner.rename(columns=columnas,inplace=True)
DATA=mergedinner.reindex(columns=columnasorden)
DATA

#### Extraccion de datos

Se generan 4 nuevas columnas (emoticones, hashtags, URLs y menciones) por cada columna de texto importante. (En este caso 8)

##### Extreaer emoticones

In [ ]:
def emojis(texto):
  if (len(texto) != 0):
    emoji_summary = adv.extract_emoji(texto)
    return emoji_summary['top_emoji']
  else:
    L=[]
    return L

In [ ]:
DATA['emojisTituloVideo']=DATA['TituloVideo'].apply(emojis)

In [ ]:
DATA['emojisTituloVideo']

In [ ]:
DATA['emojisDescripcionVideo']=DATA['DescripcionVideo'].apply(emojis)

In [ ]:
DATA['emojisDescripcionVideo']

##### Extraer URLs

In [ ]:
extractor = URLExtract()
def URLs(texto):
  URL=extractor.find_urls(texto)
  return URL

In [ ]:
DATA['URLsTituloVideo']=DATA['TituloVideo'].apply(URLs)
DATA['URLsDescripcionVideo']=DATA['DescripcionVideo'].apply(URLs)

In [ ]:
DATA['URLsTituloVideo']

In [ ]:
DATA['URLsDescripcionVideo']

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideo'].apply(codigohex)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideo'].apply(codigohex)

##### Extraer Hashtags

In [ ]:
def hastags(texto):
  pattern='#\w+'
  hastag=regexp_tokenize(texto,pattern)
  return hastag

In [ ]:
DATA['hashtagsTituloVideo']=DATA['TituloVideoModificado'].apply(hastags)
DATA['hashtagsDescripcionVideo']=DATA['DescripcionVideoModificado'].apply(hastags)

In [ ]:
DATA['hashtagsTituloVideo']

In [ ]:
DATA['hashtagsDescripcionVideo']

##### Extraer menciones o etiquetas (puede ser necesario alguna modificacion para el caso de correos)

In [ ]:
def mencion(texto):
  pattern='@\w+'
  mencion=regexp_tokenize(texto,pattern)
  return mencion

In [ ]:
DATA['MencionesTituloVideo']=DATA['TituloVideo'].apply(mencion)
DATA['MencionesDescripcionVideo']=DATA['DescripcionVideo'].apply(mencion)

In [ ]:
DATA['MencionesTituloVideo']

In [ ]:
DATA['MencionesDescripcionVideo']

##### Extraer etiquetas

In [ ]:
DATA['EtiquetasVideo']

In [ ]:
def textoetiquetas(etiquetas):
  if (str(etiquetas) != 'nan'):
    texto=" ".join(etiquetas)
  else:
    texto=''
  return texto

In [ ]:
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideo'].apply(textoetiquetas)
DATA['EtiquetasVideoTexto']

#### DATA

In [ ]:
DATA.info()

#### Limpiar texto: (Generamos una copia llamada columname_modificado de una columna a partir del cual vamos a realizar la limpieza de datos) OJO: Revisar secuencias de escape para serie tres caines y los 80

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideo'].apply(codigohex)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideo'].apply(codigohex)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideo'].apply(codigohex)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(codigohex)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar Unicode (Es aca donde se eliminan las tildes-> Solucionado) NOTA: No se entiende muy bien que realiza este parte del codigo

In [ ]:
def dropunicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropunicode)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropunicode)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropunicode)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropunicode)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar secuencias de escape

In [ ]:
DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('\r', ' '))
DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('\n', ' '))
DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('\v', ' '))
DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('\t', ' '))
DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('\f', ' '))
DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('\b', ' '))
#DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('3', 'tres')) # para tres caines
#DATA['TituloVideo'] = DATA['TituloVideo'].apply(str.replace, args=('80', 'ochenta')) # para los 80
DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('\r', ' '))
DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('\n', ' '))
DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('\v', ' '))
DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('\t', ' '))
DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('\f', ' '))
DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('\b', ' '))
#DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('3', 'tres')) # para tres caines
#DATA['TituloVideoModificado'] = DATA['TituloVideoModificado'].apply(str.replace, args=('80', 'ochenta')) # para los 80
DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('\r', ' '))
DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('\n', ' '))
DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('\v', ' '))
DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('\t', ' '))
DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('\f', ' '))
DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('\b', ' '))
#DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('3', 'tres')) # para tres caines
#DATA['DescripcionVideo'] = DATA['DescripcionVideo'].apply(str.replace, args=('80', 'ochenta')) # para los 80
DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('\r', ' '))
DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('\n', ' '))
DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('\v', ' '))
DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('\t', ' '))
DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('\f', ' '))
DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('\b', ' '))
#DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('3', 'tres')) # para tres caines
#DATA['DescripcionVideoModificado'] = DATA['DescripcionVideoModificado'].apply(str.replace, args=('80', 'ochenta')) # para los 80
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('\r', ' '))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('\n', ' '))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('\v', ' '))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('\t', ' '))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('\f', ' '))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('\b', ' '))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-mexico', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-cielos', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-chema', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-muñecas', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-munecas', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-narcos', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-cielos', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-sapos', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-chapo', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-rockers', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-masterchef', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-animados', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-prisioneros', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-caricaturas', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-años', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-anos', ''))
DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('-decada', ''))
#DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('3', 'tres')) # para tres caines
#DATA['ConsultaVideoModificado'] = DATA['ConsultaVideoModificado'].apply(str.replace, args=('80', 'ochenta')) # para los 80
DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('\r', ' '))
DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('\n', ' '))
DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('\v', ' '))
DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('\t', ' '))
DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('\f', ' '))
DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('\b', ' '))
#DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('3', 'tres')) # para tres caines
#DATA['EtiquetasVideoTexto'] = DATA['EtiquetasVideoTexto'].apply(str.replace, args=('80', 'ochenta')) # para los 80

##### Eliminar URLs

In [ ]:
def dropURLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text
def dropURL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropURLs)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropURLs)
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropURL)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropURL)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropURLs)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropURL)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar Emoticones

In [ ]:
from cleantext import clean

In [ ]:
from dataclasses import replace
def dropemoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropemoticons)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropemoticons)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropemoticons)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar emails

In [ ]:
def dropemails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropemails)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropemails)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropemails)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar Menciones

In [ ]:
def dropmenciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropmenciones)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropmenciones)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropmenciones)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Quitar caracteres repetidos


In [ ]:
def droppalabrasrepetidas(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(droppalabrasrepetidas)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(droppalabrasrepetidas)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(droppalabrasrepetidas)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar Numeros

In [ ]:
def dropnumeros(texto):
  pattern = r'[0-9]'
  texto = re.sub(pattern, '', texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropnumeros)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropnumeros)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropnumeros)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropnumeros)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar \# de Hashtags

In [ ]:
def dropnumeral(texto):
  texto = re.sub(r'#', ' ', texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropnumeral)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropnumeral)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropnumeral)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropnumeral)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Dividir palabras adjuntas (CamelCase)

In [ ]:
def splitpalabras(texto):
  texto = " ".join([s for s in re.split("([A-Z][a-z]+[^A-Z]*)",texto) if s])
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(splitpalabras)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(splitpalabras)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(splitpalabras)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(splitpalabras)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar signos de puntuacion

In [ ]:
punct = set(string.punctuation) 
punct 

In [ ]:
def puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(puntuacion)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(puntuacion)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(puntuacion)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(puntuacion)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar espacios

In [ ]:
def dropespacios(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropespacios)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropespacios)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropespacios)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropespacios)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminar tokens cortos

In [ ]:
def droppalabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(droppalabrascortas)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(droppalabrascortas)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(droppalabrascortas)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(droppalabrascortas)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Normalizar texto (Quitar mayusculas)

In [ ]:
def normatext(texto):
  textonormalizado=texto.lower()
  return textonormalizado

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(normatext)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(normatext)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(normatext)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(normatext)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Cambios particulares para cada serie

In [ ]:
def cambiosEPM(texto):
  texto1=re.sub("elpatrondelmal", "el patron del mal", texto)
  texto2=re.sub("patrondelmal", "patron del mal", texto1)
  texto3=re.sub("patrondel", "patron del", texto2)
  texto4=re.sub("pabloescobar", "pablo escobar", texto3)
  texto5=re.sub("murtes", "muertes", texto4)
  texto6=re.sub("murte", "muerte", texto5)
  texto7=re.sub("muerts", "muertes", texto6)
  texto8=re.sub("patron", " patron ", texto7)
  texto9=re.sub("escobar", " escobar ", texto8)
  texto=re.sub("pablo", " pablo ", texto9) 
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(cambiosEPM)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(cambiosEPM)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(cambiosEPM)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(cambiosEPM)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Eliminando stopwords

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
swsp=nltk.corpus.stopwords.words('spanish')

In [ ]:
def dropStopWordsSpanish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swsp:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
swpo=nltk.corpus.stopwords.words('portuguese')

In [ ]:
def dropStopWordsPortugues(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swsp:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
swen=nltk.corpus.stopwords.words('english')

In [ ]:
def dropStopWordsEnglish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swen:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropStopWordsSpanish)
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropStopWordsPortugues)
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropStopWordsEnglish)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropStopWordsSpanish)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropStopWordsPortugues)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropStopWordsEnglish)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropStopWordsSpanish)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropStopWordsPortugues)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropStopWordsEnglish)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropStopWordsSpanish)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropStopWordsPortugues)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropStopWordsEnglish)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Volvemos a eliminar exceso de espacios

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(dropespacios)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(dropespacios)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(dropespacios)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(dropespacios)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

##### Volvemos a eliminar tokens cortos

In [ ]:
DATA['TituloVideoModificado']=DATA['TituloVideoModificado'].apply(droppalabrascortas)
DATA['DescripcionVideoModificado']=DATA['DescripcionVideoModificado'].apply(droppalabrascortas)
DATA['ConsultaVideoModificado']=DATA['ConsultaVideoModificado'].apply(droppalabrascortas)
DATA['EtiquetasVideoTexto']=DATA['EtiquetasVideoTexto'].apply(droppalabrascortas)

In [ ]:
DATA['TituloVideoModificado']

In [ ]:
DATA['DescripcionVideoModificado']

In [ ]:
DATA['ConsultaVideoModificado']

In [ ]:
DATA['EtiquetasVideoTexto']

#### Estructurar y dar formato

In [ ]:
DATA.info()

In [ ]:
columnasorden=['idVideo','TituloVideo', 'TituloVideoModificado', 'emojisTituloVideo', 'URLsTituloVideo', 'hashtagsTituloVideo','MencionesTituloVideo'\
               ,'DescripcionVideo', 'DescripcionVideoModificado', 'emojisDescripcionVideo', 'URLsDescripcionVideo', 'hashtagsDescripcionVideo', 'MencionesDescripcionVideo'\
               ,'FechaPublicacionVideo','CategoriaVideo','VistasVideo','LikesVideo','FavoritoVideo','ComentariosVideo'\
               ,'EtiquetasVideo','EtiquetasVideoTexto','TransmisionVivoVideo','idCanalVideo','TituloCanalVideo','ConsultaVideo','ConsultaVideoModificado','PaisVideo','SolicitudVideo']

DATA=DATA.reindex(columns=columnasorden)
DATA

In [ ]:
DATA.info()

#### Exportar Dataframe a CSV

In [ ]:
Metadatac.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)